## VS Code 로컬 실행

프로젝트 루트 또는 capstone/ 폴더에서 실행합니다. 입력 데이터와 의존성은 README.md를 참고하세요.

현재 출력은 2026년 로컬 실행에서 다시 생성했습니다. 원본 노트북과 당시 저장 출력은 originals/에 보존되어 있습니다.


참고 문헌:
https://velog.io/@ranyjoon/%EC%82%AC%EC%9D%B4%ED%82%B7%EB%9F%B0%EC%B6%94%EC%B2%9C-%EC%8B%9C%EC%8A%A4%ED%85%9C%ED%98%91%EC%97%85-%ED%95%84%ED%84%B0%EB%A7%81-%EA%B8%B0%EB%B0%98#2-%EC%9E%A0%EC%9E%AC%EC%9A%94%EC%9D%B8-%ED%98%91%EC%97%85-%ED%95%84%ED%84%B0%EB%A7%81


In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
from pathlib import Path

DATA_CANDIDATES = (
    Path("books_review_2.csv"),
    Path("capstone") / "books_review_2.csv",
)
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "books_review_2.csv not found. Run the notebook from the project root "
        "or the capstone/ directory."
    )

df = pd.read_csv(DATA_PATH)
df.head(5)

In [3]:
ratings_matrix =df.pivot_table('review/score', index='user_num', columns='Title')

In [4]:
ratings_matrix=ratings_matrix.fillna(0)

In [5]:
# 예측 성능 평가 함수

import numpy as np

def get_rmse(R, P, Q, non_zeros):
    error = 0
    # 두개의 분해된 행렬 P와 Q.T의 내적 곱으로 예측 R 행렬 생성
    full_pred_matrix = np.dot(P, Q.T)
    
    # 실제 R 행렬에서 0이 아닌 값의 위치 인덱스 추출
    # 실제 R 행렬과 예측 행렬의 RMSE 추출
    x_non_zero = [non_zero[0] for non_zero in non_zeros]
    y_non_zero = [non_zero[1] for non_zero in non_zeros]
    R_non_zeros = R[x_non_zero, y_non_zero]
    
    full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero, y_non_zero]
      
    rmse = np.sqrt(np.mean((R_non_zeros - full_pred_matrix_non_zeros) ** 2))
    
    return rmse

In [6]:
# 행렬 분해 식 함수

def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda = 0.01):
    num_users, num_items = R.shape
    
    # P와 Q 매트릭스의 크기를 지정: 정규분포를 가진 랜덤한 값
    np.random.seed(1)
    P = np.random.normal(scale=1./K, size=(num_users, K))
    Q = np.random.normal(scale=1./K, size=(num_items, K))

    break_count = 0
       
    # R > 0 
    # 행 위치, 열 위치, 값을 non_zeros로 저장. 
    non_zeros = [ (i, j, R[i,j]) for i in range(num_users) for j in range(num_items) if R[i,j] > 0 ]
   
    # SGD기법으로 P와 Q 매트릭스를 계속 업데이트. 
    for step in range(steps):
        for i, j, r in non_zeros:
            # 실제 값과 예측 값의 차이인 오류 값 구함
            eij = r - np.dot(P[i, :], Q[j, :].T)
            # Regularization을 반영한 SGD 업데이트 공식 적용
            P[i,:] = P[i,:] + learning_rate*(eij * Q[j, :] - r_lambda*P[i,:])
            Q[j,:] = Q[j,:] + learning_rate*(eij * P[i, :] - r_lambda*Q[j,:])
       
        rmse = get_rmse(R, P, Q, non_zeros)
        if (step % 10) == 0 :
            print("### iteration step : ", step," rmse : ", rmse)
            
    return P, Q

In [7]:
# 잠재 요인 협업 필터링으로 한 예측 평점 측정

P, Q = matrix_factorization(ratings_matrix.values, K=50, steps=200, learning_rate=0.01, r_lambda = 0.01)
pred_matrix = np.dot(P, Q.T)

### iteration step :  0  rmse :  4.2974965115786805


### iteration step :  10  rmse :  2.962391810191524


### iteration step :  20  rmse :  1.0284110357465155


### iteration step :  30  rmse :  0.46150606214079054


### iteration step :  40  rmse :  0.2830170208345823


### iteration step :  50  rmse :  0.19436256144250139


... (10 outputs left out)

### iteration step :  160  rmse :  0.029632182590339003


### iteration step :  170  rmse :  0.028281101388740203


### iteration step :  180  rmse :  0.027242784861446333


### iteration step :  190  rmse :  0.026435967988572076


In [8]:
# 출력을 위해 데이터 프레임 형식으로 변환

ratings_pred_matrix = pd.DataFrame(data=pred_matrix, 
                                   index= ratings_matrix.index,
                                   columns = ratings_matrix.columns)

In [9]:
# 사용자가 관람하지 않은 추천 책 리스트 출력

def recommend_book(pred_df, ratings_matrix, userId, top=10):
    # unseen_list 만들기
    user_rating = ratings_matrix.loc[userId,:]
    already_seen = user_rating[ user_rating > 0].index.tolist()    
        
    movies_list = ratings_matrix.columns.tolist()     # 모든 영화명을 list   
    unseen_list = [ movie for movie in movies_list if movie not in already_seen]
    
    # 추천영화
    recomm_movies = pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top]
    recomm_movies_df = pd.DataFrame(data=recomm_movies.values,
                                    index=recomm_movies.index,
                                    columns=['predicted_score'])
    
    return recomm_movies_df

recommend_book(ratings_pred_matrix, ratings_matrix, 1225, top=10) # 출력 확인

,predicted_score
Title,
Riddle-Master,4.624435
Black Hole,4.391563
Mariette in Ecstasy,4.068898
Sync: The Emerging Science of Spontaneous Order,4.036290
The New Encyclopedia of the Occult,4.020941
Gramercy Park (Beeler Large Print Series),4.011989
The Sea Wolf,3.972002
Snoozers : 7 Short Short Bedtime Stories for Lively Little Kids,3.965409
What Are the Seven Wonders of the World? and 100 Other Great Cultural Lists--Fully Explicated,3.957171


In [ ]:
df[df['Title']=='The New Encyclopedia of the Occult'] # Fiction, Comics Graphic Novels, Fiction, Mathematics, 

## 홀드아웃 평점 평가

아래 평가는 사용자-책 중복 리뷰를 피벗 테이블처럼 평균 집계한 뒤, 상호작용이 2개 이상인 사용자마다 한 건을 검증용으로 남깁니다. 검증에 포함된 책은 학습 데이터에도 남도록 분할하며, 결과는 [README.md](README.md)에 기록했습니다.


In [11]:
import sys
sys.path.insert(0, str(DATA_PATH.parent.resolve()))
from evaluate_holdout import evaluate

holdout_results = evaluate(
    DATA_PATH, seed=42, factors=50, epochs=200,
    learning_rate=0.01, regularization=0.01,
)
holdout_results

{'source_review_rows': 21538,
 'user_item_pairs': 19767,
 'duplicate_user_item_pairs_aggregated': 1333,
 'train_rows': 15423,
 'test_rows': 4344,
 'users_in_test': 4344,
 'items_in_test': 1351,
 'seed': 42,
 'model_seed': 1,
 'factors': 50,
 'epochs': 200,
 'learning_rate': 0.01,
 'regularization': 0.01,
 'global_mean_rmse': 1.0753185456922765,
 'global_mean_mae': 0.8541558085520138,
 'user_mean_rmse': 1.1646994815492984,
 'user_mean_mae': 0.7902139291092745,
 'mf_raw_rmse': 1.950943188436752,
 'mf_raw_mae': 1.615654520093504,
 'mf_clipped_rmse': 1.901942947641364,
 'mf_clipped_mae': 1.5873338749782122,
 'mf_raw_prediction_min': -1.4310935105783944,
 'mf_raw_prediction_max': 5.455799889521444,
 'rating_scale': '1..5'}